In [23]:
%pip install xgboost

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 5.9 MB/s eta 0:00:00m eta 0:00:010:00:01

[notice] A new release of pip available: 22.3.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [24]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score,classification_report
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier



In [2]:
data_df = pd.read_csv("../data/feature_engineered_traffic.csv")


In [3]:
data_df.head(10)

,Day_of_week,Age_band_of_driver,Sex_of_driver,Educational_level,Vehicle_driver_relation,Driving_experience,Type_of_vehicle,Owner_of_vehicle,Service_year_of_vehicle,Defect_of_vehicle,...,Sex_of_casualty,Age_band_of_casualty,Work_of_casuality,Fitness_of_casuality,Pedestrian_movement,Cause_of_accident,Accident_severity,Hour,Time_of_day,Is_weekend
0,Monday,18-30,Male,Above high school,Employee,1-2yr,Automobile,Owner,Above 10yr,No defect,...,na,na,Unknown,Unknown,Not a Pedestrian,Moving Backward,Slight Injury,17,Evening,0
1,Monday,31-50,Male,Junior high school,Employee,Above 10yr,Public (> 45 seats),Owner,5-10yrs,No defect,...,na,na,Unknown,Unknown,Not a Pedestrian,Overtaking,Slight Injury,17,Evening,0
2,Monday,18-30,Male,Junior high school,Employee,1-2yr,Lorry (41?100Q),Owner,Unknown,No defect,...,Male,31-50,Driver,Unknown,Not a Pedestrian,Changing lane to the left,Serious Injury,17,Evening,0
3,Sunday,18-30,Male,Junior high school,Employee,5-10yr,Public (> 45 seats),Governmental,Unknown,No defect,...,Female,18-30,Driver,Normal,Not a Pedestrian,Changing lane to the right,Slight Injury,1,Night,1
4,Sunday,18-30,Male,Junior high school,Employee,2-5yr,Unknown,Owner,5-10yrs,No defect,...,na,na,Unknown,Unknown,Not a Pedestrian,Overtaking,Slight Injury,1,Night,1
5,Friday,31-50,Male,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,...,Male,31-50,Driver,Normal,Not a Pedestrian,Overloading,Slight Injury,14,Afternoon,0
6,Wednesday,18-30,Male,Junior high school,Employee,2-5yr,Automobile,Owner,Unknown,No defect,...,Female,18-30,Driver,Normal,Not a Pedestrian,Other,Slight Injury,17,Evening,0
7,Friday,18-30,Male,Junior high school,Employee,2-5yr,Automobile,Governmental,Above 10yr,No defect,...,na,na,Unknown,Normal,Not a Pedestrian,No priority to vehicle,Slight Injury,17,Evening,0
8,Friday,18-30,Male,Junior high school,Employee,Above 10yr,Lorry (41?100Q),Owner,1-2yr,No defect,...,Male,Under 18,Driver,Normal,Crossing from driver's nearside,Changing lane to the right,Slight Injury,17,Evening,0
9,Friday,18-30,Male,Junior high school,Employee,1-2yr,Automobile,Owner,2-5yrs,No defect,...,Male,18-30,Driver,Normal,Not a Pedestrian,Moving Backward,Serious Injury,17,Evening,0


In [4]:
data_df.isnull().sum()

Day_of_week                    0
Age_band_of_driver             0
Sex_of_driver                  0
Educational_level              0
Vehicle_driver_relation        0
Driving_experience             0
Type_of_vehicle                0
Owner_of_vehicle               0
Service_year_of_vehicle        0
Defect_of_vehicle              0
Area_accident_occured          0
Lanes_or_Medians               0
Road_allignment                0
Types_of_Junction              0
Road_surface_type              0
Road_surface_conditions        0
Light_conditions               0
Weather_conditions             0
Type_of_collision              0
Number_of_vehicles_involved    0
Number_of_casualties           0
Vehicle_movement               0
Casualty_class                 0
Sex_of_casualty                0
Age_band_of_casualty           0
Work_of_casuality              0
Fitness_of_casuality           0
Pedestrian_movement            0
Cause_of_accident              0
Accident_severity              0
Hour      

# seperate X and Y as features and target

In [5]:
X = data_df.drop(columns=["Accident_severity"])

In [6]:
y = data_df["Accident_severity"]

In [7]:
X.shape

(8210, 32)

In [8]:
y.shape

(8210,)

X contains the accident-related features used as inputs for the model.y contains Accident_severity, which is the outcome the model will predict.

In [9]:
X_train,X_test, y_train, y_test = train_test_split(
    X, y, test_size= 0.2, random_state= 99, stratify= y
    
)

In [10]:
 X_train.shape

(6568, 32)

In [11]:
X_test.shape

(1642, 32)

In [12]:
y_train.shape

(6568,)

In [13]:
y_test.shape

(1642,)

In [14]:
X_train.select_dtypes(include=["object"]).columns

Index(['Day_of_week', 'Age_band_of_driver', 'Sex_of_driver',
       'Educational_level', 'Vehicle_driver_relation', 'Driving_experience',
       'Type_of_vehicle', 'Owner_of_vehicle', 'Service_year_of_vehicle',
       'Defect_of_vehicle', 'Area_accident_occured', 'Lanes_or_Medians',
       'Road_allignment', 'Types_of_Junction', 'Road_surface_type',
       'Road_surface_conditions', 'Light_conditions', 'Weather_conditions',
       'Type_of_collision', 'Vehicle_movement', 'Casualty_class',
       'Sex_of_casualty', 'Age_band_of_casualty', 'Work_of_casuality',
       'Fitness_of_casuality', 'Pedestrian_movement', 'Cause_of_accident',
       'Time_of_day'],
      dtype='object')

# Encoding

In [15]:
X_train_encoded = pd.get_dummies(X_train, dtype=int)

In [16]:
X_test_encoded = pd.get_dummies(X_test, dtype=int)


In [17]:
X_test_encoded = X_test_encoded.reindex(
    columns=X_train_encoded.columns,
    fill_value=0
)

I used reindex() to align the testing columns with the training columns and filled missing encoded columns with 0.


In [43]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_encoded)

X_test_scaled = scaler.transform(X_test_encoded)

## Logistic Regression Model - 1

In [44]:
logistic_reg = LogisticRegression(

    max_iter = 1000,
    random_state =99,
    class_weight = "balanced"
)

In [46]:
logistic_reg.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,99
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


# predictions

In [48]:
y_predict = logistic_reg.predict(X_test_scaled)

In [49]:
y_predict

array(['Slight Injury', 'Slight Injury', 'Fatal injury', ...,
       'Slight Injury', 'Slight Injury', 'Slight Injury'],
      shape=(1642,), dtype=object)

## Performance metric

In [51]:
accuracy = accuracy_score(y_test, y_predict)

In [52]:
accuracy

0.523142509135201

In [66]:
precision = precision_score(y_test, y_predict, average="weighted")

In [67]:
precision

0.7635727576290978

In [69]:
recall = recall_score(y_test, y_predict, average="weighted", )
f1 = f1_score(y_test, y_predict, average="weighted", )
recall, f1

(0.523142509135201, 0.6051622905405438)

In [70]:
print(classification_report(
    y_test,
    y_predict,
))

                precision    recall  f1-score   support

  Fatal injury       0.04      0.35      0.07        17
Serious Injury       0.12      0.34      0.18       209
 Slight Injury       0.87      0.55      0.67      1416

      accuracy                           0.52      1642
     macro avg       0.34      0.42      0.31      1642
  weighted avg       0.76      0.52      0.61      1642



a baseline model to classify road traffic accidents into Slight Injury, Serious Injury, and Fatal injury. I used StandardScaler for feature scaling and class_weight="balanced" to handle class imbalance. The model achieved 52.31% accuracy and a weighted average F1-score of 60%, indicating difficulty in identifying minority classes.


## Random Forest Classifier - 2

In [73]:
random_forest = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=99
)

random_forest.fit(X_train_encoded, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


I implemented Random Forest Classifier to classify road traffic accidents into Slight Injury, Serious Injury, and Fatal injury. I used 200 decision trees and balanced class weights to address class imbalance. Unlike LogistIts performance will be compared with Logistic Regression and XGBoost.

In [74]:
y_pred_ =  random_forest.predict(X_test_encoded)

In [75]:
y_pred_

array(['Slight Injury', 'Slight Injury', 'Slight Injury', ...,
       'Slight Injury', 'Slight Injury', 'Slight Injury'],
      shape=(1642,), dtype=object)

In [83]:
rf_accuracy = accuracy_score(y_test, y_pred_)
rf_precision = precision_score(y_test, y_pred_, average="weighted")
rf_recall = recall_score(y_test, y_pred_, average="weighted")
rf_f1 = f1_score(y_test, y_pred_, average="weighted")

/Users/babitaadhikari/traffic/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [78]:
rf_accuracy

0.8623629719853837

In [79]:
rf_precision

0.7436698954514637

In [80]:
rf_recall

0.8623629719853837

In [81]:
rf_f1

0.798630456724201

In [85]:
print(classification_report(
    y_test,
    y_pred_,
))

                precision    recall  f1-score   support

  Fatal injury       0.00      0.00      0.00        17
Serious Injury       0.00      0.00      0.00       209
 Slight Injury       0.86      1.00      0.93      1416

      accuracy                           0.86      1642
     macro avg       0.29      0.33      0.31      1642
  weighted avg       0.74      0.86      0.80      1642



/Users/babitaadhikari/traffic/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/babitaadhikari/traffic/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/babitaadhikari/traffic/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

## XGBoost - 3

In [18]:
from sklearn.preprocessing import LabelEncoder


In [19]:
label_encoder = LabelEncoder()


In [20]:
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)


In [21]:
label_encoder.classes_

array(['Fatal injury', 'Serious Injury', 'Slight Injury'], dtype=object)

In [25]:
xgb_model = XGBClassifier(
    n_estimators=200,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=99
)

In [26]:
xgb_model.fit(X_train_encoded, y_train_encoded)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [27]:
y_xgboost = xgb_model.predict(X_test_encoded)

In [28]:
y_xgboost

array([2, 2, 2, ..., 2, 2, 2], shape=(1642,))

In [29]:
xgb_accuracy = accuracy_score(y_test_encoded, y_xgboost)

In [30]:
xgb_accuracy

0.8617539585870889

In [31]:
xgb_precision = precision_score(y_test_encoded, y_xgboost, average="weighted")


In [32]:
xgb_precision

0.8168605359317905

In [33]:
xgb_recall = recall_score(y_test_encoded, y_xgboost, average="weighted")

In [34]:
xgb_recall

0.8617539585870889

In [35]:
xgb_f1 = f1_score(y_test_encoded, y_xgboost, average="weighted")


In [36]:
xgb_f1

0.8178745378961919